In [1]:
import pandas as pd

In [2]:
orders = pd.DataFrame({
    "order_id": ["O101", "O102", "O103", "O104", "O105"],
    "customer_id": ["C01", "C02", "C03", "C04", "C05"],
    "amount": [1200, 800, 1500, 600, 2000],
    "status": ["Completed", "Completed", "Completed", "Cancelled", "Completed"]
})

In [3]:
payments = pd.DataFrame({
    "payment_id": ["P501", "P502", "P503", "P504", "P505"],
    "order_id": ["O101", "O103", "O104", "O105", "O999"],
    "amount": [1200, 1500, 600, 1800, 700],
    "status": ["Success", "Success", "Refunded", "Success", "Success"]
})

In [4]:
orders.head()

,order_id,customer_id,amount,status
0,O101,C01,1200,Completed
1,O102,C02,800,Completed
2,O103,C03,1500,Completed
3,O104,C04,600,Cancelled
4,O105,C05,2000,Completed


In [5]:
payments.head()

,payment_id,order_id,amount,status
0,P501,O101,1200,Success
1,P502,O103,1500,Success
2,P503,O104,600,Refunded
3,P504,O105,1800,Success
4,P505,O999,700,Success


Return only records whose order_id exists in both datasets.

In [17]:
pd.merge(left=orders, right=payments, how="inner", on="order_id")

,order_id,customer_id,amount_x,status_x,payment_id,amount_y,status_y
0,O101,C01,1200,Completed,P501,1200,Success
1,O103,C03,1500,Completed,P502,1500,Success
2,O104,C04,600,Cancelled,P503,600,Refunded
3,O105,C05,2000,Completed,P504,1800,Success


Keep all orders and attach payment information wherever available. Make the overlapping amount and status column names meaningful rather than _x / _y.

In [7]:
pd.merge(
    left=orders, 
    right=payments, 
    how="left", 
    on="order_id",
    suffixes=("_orders", "_payments")
)

,order_id,customer_id,amount_orders,status_orders,payment_id,amount_payments,status_payments
0,O101,C01,1200,Completed,P501,1200.0,Success
1,O102,C02,800,Completed,NaN,NaN,NaN
2,O103,C03,1500,Completed,P502,1500.0,Success
3,O104,C04,600,Cancelled,P503,600.0,Refunded
4,O105,C05,2000,Completed,P504,1800.0,Success


Produce one reconciliation dataset containing all records from both systems and use Pandas' merge indicator to distinguish:

- both
- left_only
- right_only

In [9]:
all_rec = pd.merge(
    left=orders,
    right=payments,
    how="outer",
    on="order_id",
    suffixes=("_orders", "_payments"),
    indicator=True
)

In [10]:
all_rec

,order_id,customer_id,amount_orders,status_orders,payment_id,amount_payments,status_payments,_merge
0,O101,C01,1200.0,Completed,P501,1200.0,Success,both
1,O102,C02,800.0,Completed,NaN,NaN,NaN,left_only
2,O103,C03,1500.0,Completed,P502,1500.0,Success,both
3,O104,C04,600.0,Cancelled,P503,600.0,Refunded,both
4,O105,C05,2000.0,Completed,P504,1800.0,Success,both
5,O999,NaN,NaN,NaN,P505,700.0,Success,right_only


From that result, return only orders that don't have any payment record.

In [11]:
all_rec[all_rec['_merge'].isin(['left_only'])]

,order_id,customer_id,amount_orders,status_orders,payment_id,amount_payments,status_payments,_merge
1,O102,C02,800.0,Completed,NaN,NaN,NaN,left_only


Return only orphan payment records—payments whose order_id doesn't exist in orders.

In [12]:
all_rec[all_rec['_merge'].isin(['right_only'])]

,order_id,customer_id,amount_orders,status_orders,payment_id,amount_payments,status_payments,_merge
5,O999,NaN,NaN,NaN,P505,700.0,Success,right_only


In [16]:
pd.merge(
    left=orders,
    right=payments,
    how="outer",
    on="order_id",
    suffixes=("_orders", "_payments"),
    indicator=True,
    validate="1:1"
)

,order_id,customer_id,amount_orders,status_orders,payment_id,amount_payments,status_payments,_merge
0,O101,C01,1200.0,Completed,P501,1200.0,Success,both
1,O102,C02,800.0,Completed,NaN,NaN,NaN,left_only
2,O103,C03,1500.0,Completed,P502,1500.0,Success,both
3,O104,C04,600.0,Cancelled,P503,600.0,Refunded,both
4,O105,C05,2000.0,Completed,P504,1800.0,Success,both
5,O999,NaN,NaN,NaN,P505,700.0,Success,right_only
